In [1]:
%%writefile cronjob_eficiente.yaml
apiVersion: batch/v1
kind: CronJob
metadata:
  name: cronjob-eficiente-batch
  namespace: default
spec:
  schedule: "*/5 * * * *" # <-- ¡Aquí está la única pieza nueva! Cada 5 minutos.
  concurrencyPolicy: Forbid # <-- Si un proceso se traba, prohíbe que se encime el siguiente.
  jobTemplate:
    spec:
      template:
        spec:
          restartPolicy: OnFailure
          containers:
          - name: python-worker
            image: python:3.10-slim
            command: ["python", "/app/procesador.py"]
            env:
            - name: APP_ENV
              value: "Produccion-Costos-Bajos"
            resources:
              requests:
                memory: "64Mi"
                cpu: "100m"
              limits:
                memory: "128Mi"
                cpu: "200m"
            volumeMounts:
            - name: espacio-codigo
              mountPath: /app
          volumes:
          - name: espacio-codigo
            configMap:
              name: procesador-config

Writing cronjob_eficiente.yaml


In [1]:
%%bash
kubectl apply -f cronjob_eficiente.yaml

cronjob.batch/cronjob-eficiente-batch created


In [2]:
%%bash
kubectl get cronjobs

NAME                      SCHEDULE      SUSPEND   ACTIVE   LAST SCHEDULE   AGE
cronjob-eficiente-batch   */5 * * * *   False     0        <none>          6s


In [3]:
%%bash
kubectl get jobs

NAME                   COMPLETIONS   DURATION   AGE
batch-job-inicial      1/1           14s        3d6h
batch-telemetria-job   1/1           7s         3d4h
job-eficiente-batch    1/1           12s        2m32s
job-pathlib-batch      1/1           9s         29h


In [5]:
%%bash
kubectl logs job/cronjob-eficiente-batch

Error from server (NotFound): jobs.batch "cronjob-eficiente-batch" not found


CalledProcessError: Command 'b'kubectl logs job/cronjob-eficiente-batch\n'' returned non-zero exit status 1.

In [9]:
import os
print(os.getcwd())

/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/eighth_container/dia_3/eje_4


In [6]:
%%bash
kubectl delete cronjob cronjob-eficiente-batch

cronjob.batch "cronjob-eficiente-batch" deleted


In [11]:
%%writefile reporte.py
import os
import time
import sys
from pathlib import Path

def generar_reporte():
    # 1. Leemos el entorno (por defecto será Desarrollo-Local)
    app_env = os.getenv("APP_ENV", "Desarrollo-Local")
    
    # 2. Definimos la ruta de la carpeta interna donde el script intentará escribir
    ruta_carpeta = Path("/app/datos_reporte")
    archivo_reporte = ruta_carpeta / "reporte_salud.txt"
    
    print(f"=== INICIANDO VALIDACIÓN EN ENTORNO: {app_env} ===")
    
    # 3. Verificamos si la carpeta existe. Si no, la creamos.
    if not ruta_carpeta.exists():
        print(f"📁 La carpeta '{ruta_carpeta}' no existe. Creándola...")
        try:
            ruta_carpeta.mkdir(parents=True, exist_ok=True)
        except Exception as e:
            print(f"❌ Error al crear la carpeta en el disco: {e}")
            sys.exit(1)
    else:
        print(f"📁 La carpeta '{ruta_carpeta}' ya existe.")
        
    time.sleep(1)
    
    # 4. Intentamos escribir el archivo de texto con los datos
    try:
        with open(archivo_reporte, "w") as f:
            f.write(f"Reporte de salud para: {app_env}\n")
            f.write("Estado: Conexión de almacenamiento exitosa.\n")
        print(f"✅ Archivo escrito con éxito en: {archivo_reporte}")
    except Exception as e:
        print(f"❌ Error al escribir el archivo: {e}")
        sys.exit(1)
        
    print("=== TRABAJO FINALIZADO ===")

if __name__ == "__main__":
    generar_reporte()

Writing reporte.py


In [7]:
!python3 reporte.py

=== INICIANDO VALIDACIÓN EN ENTORNO: Desarrollo-Local ===
📁 La carpeta '/app/datos_reporte' no existe. Creándola...
❌ Error al crear la carpeta en el disco: [Errno 30] Read-only file system: '/app'


In [8]:
%%bash
kubectl delete configmap procesador-config --ignore-not-found=true
kubectl create configmap procesador-config --from-file=reporte.py

configmap "procesador-config" deleted
configmap/procesador-config created


In [14]:
%%writefile job_almacenamiento.yaml
apiVersion: batch/v1
kind: Job
metadata:
  name: job-almacenamiento-batch
spec:
  backoffLimit: 1
  template:
    spec:
      restartPolicy: Never
      containers:
      - name: python-worker
        image: python:3.10-slim
        command: ["python", "/app/reporte.py"]
        env:
        - name: APP_ENV
          value: "Produccion-Almacenamiento-Fijo"
        volumeMounts:
        # Pestaña 1: Montamos el código (lo que ya sabes hacer)
        - name: espacio-codigo
          mountPath: /app
        # Pestaña 2: ¡El nuevo puente de salida!
        - name: disco-local-computadora
          mountPath: /app/datos_reporte # <-- La carpeta que Python busca
      volumes:
      # Origen 1: El código del ConfigMap
      - name: espacio-codigo
        configMap:
          name: procesador-config
      # Origen 2: Carpeta física real en tu computadora
      - name: disco-local-computadora
        hostPath:
          path: /run/desktop/mnt/host/wsl/datos_locales # Estándar de Docker Desktop para mapear tu disco
          type: DirectoryOrCreate # <-- Si la carpeta no existe, Kubernetes la crea físicamente

Writing job_almacenamiento.yaml


In [9]:
%%bash
kubectl apply -f job_almacenamiento.yaml

job.batch/job-almacenamiento-batch created


In [10]:
%%bash
kubectl get jobs

NAME                       COMPLETIONS   DURATION   AGE
batch-job-inicial          1/1           14s        3d6h
batch-telemetria-job       1/1           7s         3d4h
job-almacenamiento-batch   0/1           4s         4s
job-eficiente-batch        1/1           12s        4m11s
job-pathlib-batch          1/1           9s         29h


In [11]:
%%bash
kubectl logs job/job-almacenamiento-batch

=== INICIANDO VALIDACIÓN EN ENTORNO: Produccion-Almacenamiento-Fijo ===
📁 La carpeta '/app/datos_reporte' ya existe.
✅ Archivo escrito con éxito en: /app/datos_reporte/reporte_salud.txt
=== TRABAJO FINALIZADO ===


In [12]:
!ls -l /run/desktop/mnt/host/wsl/datos_locales

ls: /run/desktop/mnt/host/wsl/datos_locales: No such file or directory


In [13]:
%%bash
kubectl run inspector-almacenamiento --rm -i --restart=Never --image=python:3.10-slim --overrides='
{
  "spec": {
    "volumes": [
      {
        "name": "disco-real",
        "hostPath": { "path": "/run/desktop/mnt/host/wsl/datos_locales" }
      }
    ],
    "containers": [
      {
        "name": "inspector",
        "image": "python:3.10-slim",
        "command": ["cat", "/mnt/reporte_salud.txt"],
        "volumeMounts": [
          { "name": "disco-real", "mountPath": "/mnt" }
        ]
      }
    ]
  }
}'

Reporte de salud para: Produccion-Almacenamiento-Fijo
Estado: Conexión de almacenamiento exitosa.
pod "inspector-almacenamiento" deleted


In [14]:
%%bash
kubectl delete job job-almacenamiento-batch

job.batch "job-almacenamiento-batch" deleted
